# Ablation Studies

## Experiments:
1. **θ_BS dimension**: {8, 16, 32, 64, 128, 256}
2. **Site injection type**: FiLM vs concat vs add
3. **Number of pre-training BSs**: {2, 4, 6}
4. **Cold start analysis**: NMSE vs number of training samples

In [ ]:
import sys, os
from pathlib import Path

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import json
import numpy as np
import matplotlib.pyplot as plt

device = 'cuda'
SAVE_PREFIX = 'phase3'
CKPT_DIR = Path(f'{PROJECT_ROOT}/assets/checkpoints/{SAVE_PREFIX}')
print(f'Checkpoint dir: {CKPT_DIR}')

In [ ]:
TEST_BS_ID = 6

## 1. θ_BS Dimension Ablation

In [ ]:
# Load dimension ablation results from phase3 step 3-1
dim_path = CKPT_DIR / '3-1_dim_results.json'
if dim_path.exists():
    with open(dim_path) as f:
        dim_results = json.load(f)
    DIMS = sorted([int(k) for k in dim_results.keys()])
    vals = [dim_results[str(d)] for d in DIMS]
    print(f'Loaded dim results: {dict(zip(DIMS, [f"{v:.2f}" for v in vals]))}')

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(DIMS, vals, 'o-', linewidth=2, markersize=8)
    ax.set_xlabel('θ_BS dimension')
    ax.set_ylabel('NMSE (dB)')
    ax.set_title('Effect of Site Embedding Dimension')
    ax.set_xscale('log', base=2)
    ax.set_xticks(DIMS)
    ax.set_xticklabels(DIMS)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    best_dim = DIMS[np.argmin(vals)]
    print(f'Best dim: {best_dim} ({min(vals):.2f} dB)')
else:
    print(f'Not found: {dim_path}')
    print('Run: python -m src.experiments.3_ablation.train --step 3-1')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
if dim_path.exists():
    ax.plot(DIMS, vals, 'o-', linewidth=2, markersize=8, color='C0')
    ax.set_xlabel('θ_BS dimension')
    ax.set_ylabel('NMSE (dB)')
    ax.set_title('Effect of Site Embedding Dimension')
    ax.set_xscale('log', base=2)
    ax.set_xticks(DIMS)
    ax.set_xticklabels(DIMS)
    ax.grid(True, alpha=0.3)
    # Annotate best
    best_idx = np.argmin(vals)
    ax.annotate(f'Best: dim={DIMS[best_idx]}', xy=(DIMS[best_idx], vals[best_idx]),
                xytext=(10, 10), textcoords='offset points', fontsize=9,
                arrowprops=dict(arrowstyle='->', color='gray'))
plt.tight_layout()
plt.show()

## 2. Site Injection Type

In [ ]:
# Load injection type results from phase3 step 3-2
inject_path = CKPT_DIR / '3-2_injection_results.json'
if inject_path.exists():
    with open(inject_path) as f:
        inject_results = json.load(f)
    print(f'Loaded injection results: {inject_results}')

    types = list(inject_results.keys())
    vals = [inject_results[t] for t in types]

    fig, ax = plt.subplots(figsize=(7, 4))
    colors = ['C0' if t == 'film' else 'C1' for t in types]
    bars = ax.bar(types, vals, color=colors, edgecolor='black', linewidth=0.5)
    ax.set_xlabel('Site Injection Type')
    ax.set_ylabel('NMSE (dB)')
    ax.set_title('Effect of Site Injection Method')
    ax.grid(True, alpha=0.3, axis='y')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v - 0.5, f'{v:.1f}',
                ha='center', va='top', fontsize=9, fontweight='bold', color='white')
    plt.tight_layout()
    plt.show()
else:
    print(f'Not found: {inject_path}')
    print('Run: python -m src.experiments.3_ablation.train --step 3-2')

## 3. Cold Start Analysis

In [ ]:
# Load cold start results from phase3 step 3-5
cold_path = CKPT_DIR / '3-5_cold_start_results.json'
if cold_path.exists():
    with open(cold_path) as f:
        cold_start = json.load(f)
    print(f'Loaded cold start results')

    fig, ax = plt.subplots(figsize=(8, 5))
    for method, color, label in [
        ('theta_bs_adapt', 'C0', 'θ_BS adapt'),
        ('from_scratch', 'C2', 'From scratch'),
    ]:
        counts = sorted([int(k) for k in cold_start[method].keys()])
        means = [cold_start[method][str(n)]['mean'] for n in counts]
        stds = [cold_start[method][str(n)]['std'] for n in counts]
        ax.errorbar(counts, means, yerr=stds, marker='o', label=label,
                    color=color, capsize=3, linewidth=2)

    ax.set_xlabel('Number of training samples')
    ax.set_ylabel('NMSE (dB)')
    ax.set_title(f'Cold Start: Adaptation Speed for BS{TEST_BS_ID}')
    ax.set_xscale('log')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Find crossover
    adapt_means = [cold_start['theta_bs_adapt'][str(n)]['mean'] for n in counts]
    scratch_means = [cold_start['from_scratch'][str(n)]['mean'] for n in counts]
    for i, n in enumerate(counts):
        if scratch_means[i] <= adapt_means[i]:
            ax.axvline(x=n, color='gray', linestyle='--', alpha=0.5)
            ax.annotate(f'Crossover ~{n}', xy=(n, adapt_means[i]),
                        fontsize=9, color='gray')
            break

    plt.tight_layout()
    plt.show()
else:
    print(f'Not found: {cold_path}')
    print('Run: python -m src.experiments.3_ablation.train --step 3-5')